# Predict on Local Machine
Sử dụng model đã train để dự đoán trên toàn bộ dataset

**Yêu cầu**: 
- Đã huấn luyện model (chạy notebook 02.train_CNN_PyTorch_local.ipynb)
- Có file `model_cnn_pytorch_full.pt`

In [ ]:
import os
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import json

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Load Model

In [ ]:
# Định nghĩa CNN model (giống như khi training)
class CNNClassifier(nn.Module):
    def __init__(self, input_size, num_classes=8):
        super(CNNClassifier, self).__init__()
        
        # Conv blocks
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(2)
        self.drop1 = nn.Dropout(0.25)
        
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.conv4 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(2)
        self.drop2 = nn.Dropout(0.25)
        
        self.conv5 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm1d(256)
        self.conv6 = nn.Conv1d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm1d(256)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.drop3 = nn.Dropout(0.25)
        
        # FC layers
        self.fc1 = nn.Linear(256, 128)
        self.bn_fc1 = nn.BatchNorm1d(128)
        self.drop_fc1 = nn.Dropout(0.5)
        
        self.fc2 = nn.Linear(128, 64)
        self.bn_fc2 = nn.BatchNorm1d(64)
        self.drop_fc2 = nn.Dropout(0.5)
        
        self.fc3 = nn.Linear(64, num_classes)
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.drop1(x)
        
        # Block 2
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.drop2(x)
        
        # Block 3
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.relu(self.bn6(self.conv6(x)))
        x = self.global_avg_pool(x)
        x = x.squeeze(-1)
        x = self.drop3(x)
        
        # FC layers
        x = self.relu(self.bn_fc1(self.fc1(x)))
        x = self.drop_fc1(x)
        x = self.relu(self.bn_fc2(self.fc2(x)))
        x = self.drop_fc2(x)
        x = self.fc3(x)
        
        return x

print("✅ Model class defined")

In [ ]:
# Load model
print("📥 Load model...")
model_path = 'model_cnn_pytorch_full.pt'

if not os.path.exists(model_path):
    raise FileNotFoundError(f"❌ Model file '{model_path}' not found!")

model_info = torch.load(model_path, map_location=device)
num_classes = model_info['num_classes']
input_size = model_info['input_size']
label_mapping = model_info['label_mapping']
mean = model_info['mean']
std = model_info['std']
test_accuracy = model_info['test_accuracy']
test_loss = model_info['test_loss']

# Khởi tạo model
model = CNNClassifier(input_size=input_size, num_classes=num_classes)
model.load_state_dict(model_info['state_dict'])
model = model.to(device)
model.eval()

print(f"✅ Model loaded")
print(f"   Num classes: {num_classes}")
print(f"   Input size: {input_size}")
print(f"   Test Accuracy: {test_accuracy:.2f}%")
print(f"\n📋 Label mapping:")
for label, idx in sorted(label_mapping.items(), key=lambda x: x[1]):
    print(f"   {idx}: {label}")

## Load Data từ Server

In [ ]:
# Load dữ liệu
data_dir = "data_for_training"

print("📥 Load dữ liệu...")
average_ndvi = xr.open_dataarray(os.path.join(data_dir, "average_ndvi.nc"))
average_vv = xr.open_dataarray(os.path.join(data_dir, "average_vv.nc"))
average_vh = xr.open_dataarray(os.path.join(data_dir, "average_vh.nc"))

print(f"✅ NDVI shape: {average_ndvi.shape}")
print(f"✅ VV shape: {average_vv.shape}")
print(f"✅ VH shape: {average_vh.shape}")

# Hiển thị thông tin
print(f"\n📊 Spatial coordinates:")
print(f"   X: {average_ndvi.x.values.min():.4f} to {average_ndvi.x.values.max():.4f}")
print(f"   Y: {average_ndvi.y.values.min():.4f} to {average_ndvi.y.values.max():.4f}")

## Predict trên toàn bộ Dataset

In [ ]:
# Chuẩn bị dữ liệu prediction
print("🔧 Chuẩn bị dữ liệu prediction...")

# Reshape dữ liệu thành grid
height = average_ndvi.shape[1]
width = average_ndvi.shape[2]

X_pred = []
for i in range(height):
    for j in range(width):
        # Lấy giá trị từ mỗi band tại vị trí (i, j)
        ndvi_val = average_ndvi.values[:, i, j]
        vv_val = average_vv.values[:, i, j]
        vh_val = average_vh.values[:, i, j]
        
        # Kết hợp các band
        pixel_data = np.concatenate((ndvi_val, vv_val, vh_val))
        X_pred.append(pixel_data)

X_pred = np.array(X_pred)
print(f"✅ Prediction data shape: {X_pred.shape}")

In [ ]:
# Normalize dữ liệu
print("🔧 Normalize dữ liệu...")
X_pred_normalized = (X_pred - mean) / (std + 1e-8)
print(f"✅ Data normalized")

In [ ]:
# Convert to tensor và predict
print("🚀 Predict...")
X_pred_tensor = torch.FloatTensor(X_pred_normalized).unsqueeze(1)

pred_dataset = TensorDataset(X_pred_tensor)
pred_loader = DataLoader(pred_dataset, batch_size=128, shuffle=False)

predictions = []
with torch.no_grad():
    for batch_idx, (X_batch,) in enumerate(pred_loader):
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        predictions.extend(predicted.cpu().numpy())
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {min((batch_idx + 1) * 128, len(X_pred))} / {len(X_pred)} pixels")

predictions = np.array(predictions)
print(f"\n✅ Prediction completed!")
print(f"   Total predictions: {len(predictions)}")
print(f"   Class distribution: {np.bincount(predictions)}")

In [ ]:
# Reshape predictions thành spatial grid
print("🔧 Reshape predictions...")
pred_map = predictions.reshape(height, width)
print(f"✅ Prediction map shape: {pred_map.shape}")

In [ ]:
# Tạo xarray DataArray để dễ lưu
print("🔧 Tạo xarray DataArray...")
pred_xarray = xr.DataArray(
    pred_map,
    coords={'y': average_ndvi.y.values, 'x': average_ndvi.x.values},
    dims=['y', 'x'],
    name='land_use_class'
)

# Copy CRS từ original data
if hasattr(average_ndvi, 'rio'):
    pred_xarray = pred_xarray.rio.write_crs(average_ndvi.rio.crs)

print(f"✅ DataArray created")
print(f"   Shape: {pred_xarray.shape}")
if hasattr(pred_xarray, 'rio') and pred_xarray.rio.crs:
    print(f"   CRS: {pred_xarray.rio.crs}")

## Visualize và Save Results

In [ ]:
# Visualize prediction map
print("📊 Visualize prediction map...")
fig, ax = plt.subplots(figsize=(12, 10))

# Reverse label mapping để display
idx_to_label = {v: k for k, v in label_mapping.items()}

# Vẽ
im = ax.imshow(pred_map, cmap='tab10', interpolation='nearest')
ax.set_title('Predicted Land Use Classification', fontsize=14, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Class Index')

plt.tight_layout()
plt.savefig('prediction_map.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Prediction map saved: prediction_map.png")

In [ ]:
# Save prediction as NetCDF
print("💾 Save prediction as NetCDF...")
output_file = 'land_use_prediction.nc'
pred_xarray.to_netcdf(output_file)
print(f"✅ Saved: {output_file}")
print(f"   Size: {os.path.getsize(output_file) / 1024**2:.2f} MB")

In [ ]:
# Save prediction as GeoTIFF (nếu có rasterio)
try:
    import rasterio
    from rasterio.transform import Affine
    
    print("💾 Save prediction as GeoTIFF...")
    output_tiff = 'land_use_prediction.tif'
    
    # Calculate transform
    x_res = (average_ndvi.x.values[1] - average_ndvi.x.values[0])
    y_res = (average_ndvi.y.values[1] - average_ndvi.y.values[0])
    x_min = average_ndvi.x.values[0] - x_res / 2
    y_max = average_ndvi.y.values[0] - y_res / 2
    
    transform = Affine.translation(x_min, y_max) * Affine.scale(x_res, y_res)
    
    # Write to GeoTIFF
    with rasterio.open(
        output_tiff, 'w',
        driver='GTiff',
        height=pred_map.shape[0],
        width=pred_map.shape[1],
        count=1,
        dtype=pred_map.dtype,
        transform=transform,
        crs='EPSG:32648'  # Thay đổi CRS nếu cần
    ) as dst:
        dst.write(pred_map, 1)
    
    print(f"✅ Saved: {output_tiff}")
    print(f"   Size: {os.path.getsize(output_tiff) / 1024**2:.2f} MB")
except ImportError:
    print("⚠️  rasterio not installed, skipping GeoTIFF export")

In [ ]:
# Lưu metadata
print("💾 Save metadata...")
metadata = {
    'model_type': 'CNN PyTorch',
    'num_classes': num_classes,
    'label_mapping': label_mapping,
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'prediction_map_shape': pred_map.shape,
    'class_distribution': {int(k): int(v) for k, v in zip(*np.unique(pred_map, return_counts=True))},
    'x_range': [float(average_ndvi.x.values.min()), float(average_ndvi.x.values.max())],
    'y_range': [float(average_ndvi.y.values.min()), float(average_ndvi.y.values.max())]
}

import json
with open('prediction_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata saved: prediction_metadata.json")
print(json.dumps(metadata, indent=2))

In [ ]:
print("\n" + "="*50)
print("✅ PREDICTION COMPLETED!")
print("="*50)
print(f"\n📁 Output files:")
print(f"  1. land_use_prediction.nc (NetCDF)")
print(f"  2. land_use_prediction.tif (GeoTIFF) - if rasterio available")
print(f"  3. prediction_map.png (visualization)")
print(f"  4. prediction_metadata.json (metadata)")
print(f"\n🚀 Next steps: Upload these files back to server if needed")